# From Raw Data to Trusted Analytics: NYC Taxi Data Pipeline
## Teresa Ferrill  
#### October 8, 2026

This notebook inspects January 2025 NYC yellow-taxi trip records and the taxi-zone lookup table. It examines data types, missing values, and potential data quality issues to guide the development of a repeatable analytics pipeline.     

### 1. Install required packages

In [1]:
%pip install pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


### 2. Load the files
*Note - run the notebook from the notebooks folder*

In [2]:
from pathlib import Path
import pandas as pd

raw_dir = Path("../data/raw")

trips = pd.read_parquet(raw_dir / "yellow_tripdata_2025-01.parquet")
zones = pd.read_csv(raw_dir / "taxi_zone_lookup.csv")

print(f"Trips: {len(trips):,}")
print(f"Taxi zones: {len(zones):,}")

display(trips.head())
display(zones.head())

Trips: 3,475,226
Taxi zones: 265


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


### 3. Inspect the columns and missing values

In [3]:
trips.info()

missing_summary = pd.DataFrame({
    "missing_count": trips.isna().sum(),
    "missing_percent": trips.isna().mean().mul(100).round(2)
})

display(missing_summary.sort_values("missing_count", ascending=False))
display(trips.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

,missing_count,missing_percent
Airport_fee,540149,15.54
passenger_count,540149,15.54
congestion_surcharge,540149,15.54
RatecodeID,540149,15.54
store_and_fwd_flag,540149,15.54
VendorID,0,0.00
mta_tax,0,0.00
total_amount,0,0.00
improvement_surcharge,0,0.00
tolls_amount,0,0.00


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
count,3.475226e+06,3475226,3475226,2.935077e+06,3.475226e+06,2.935077e+06,3.475226e+06,3.475226e+06,3.475226e+06,3.475226e+06,3.475226e+06,3.475226e+06,3.475226e+06,3.475226e+06,3.475226e+06,3.475226e+06,2.935077e+06,2.935077e+06,3.475226e+06
mean,1.785428e+00,2025-01-17 11:02:55.910964,2025-01-17 11:17:56.997901,1.297859e+00,5.855126e+00,2.482535e+00,1.651916e+02,1.641252e+02,1.036623e+00,1.708180e+01,1.317737e+00,4.780991e-01,2.959813e+00,4.493081e-01,9.547946e-01,2.561129e+01,2.225237e+00,1.239111e-01,4.834093e-01
min,1.000000e+00,2024-12-31 20:47:55,2024-12-18 07:52:40,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,-9.000000e+02,-7.500000e+00,-5.000000e-01,-8.600000e+01,-1.269400e+02,-1.000000e+00,-9.010000e+02,-2.500000e+00,-1.750000e+00,-7.500000e-01
25%,2.000000e+00,2025-01-10 07:59:01,2025-01-10 08:15:29.500000,1.000000e+00,9.800000e-01,1.000000e+00,1.320000e+02,1.130000e+02,1.000000e+00,8.600000e+00,0.000000e+00,5.000000e-01,0.000000e+00,0.000000e+00,1.000000e+00,1.520000e+01,2.500000e+00,0.000000e+00,0.000000e+00
50%,2.000000e+00,2025-01-17 15:41:33,2025-01-17 15:59:34,1.000000e+00,1.670000e+00,1.000000e+00,1.620000e+02,1.620000e+02,1.000000e+00,1.211000e+01,0.000000e+00,5.000000e-01,2.450000e+00,0.000000e+00,1.000000e+00,1.995000e+01,2.500000e+00,0.000000e+00,7.500000e-01
75%,2.000000e+00,2025-01-24 19:34:06,2025-01-24 19:48:31,1.000000e+00,3.100000e+00,1.000000e+00,2.340000e+02,2.340000e+02,1.000000e+00,1.950000e+01,2.500000e+00,5.000000e-01,3.930000e+00,0.000000e+00,1.000000e+00,2.778000e+01,2.500000e+00,0.000000e+00,7.500000e-01
max,7.000000e+00,2025-02-01 00:00:44,2025-02-01 23:44:11,9.000000e+00,2.764236e+05,9.900000e+01,2.650000e+02,2.650000e+02,5.000000e+00,8.633721e+05,1.500000e+01,1.050000e+01,4.000000e+02,1.709400e+02,1.000000e+00,8.633804e+05,2.500000e+00,6.750000e+00,7.500000e-01
std,4.263282e-01,NaN,NaN,7.507503e-01,5.646016e+02,1.163277e+01,6.452948e+01,6.940169e+01,7.013334e-01,4.634729e+02,1.861509e+00,1.374623e-01,3.779681e+00,2.002582e+00,2.781938e-01,4.636585e+02,9.039932e-01,4.725090e-01,3.619307e-01


### Findings 
- Five columns have the same missing-value count. Check whether those values are missing in the same records before assuming a shared cause.  
- Pickup dates extend outside January 2025.  
- Maximum distance is approximately 276,424 miles, and maximum fare is approximately $863,372. These warrant investigation.  
- Negative fares and zero-distance trips should be flagged for review before deciding whether to exclude them.  

### 4. Check trip durations and distances

In [4]:
duration_minutes = (
    trips["tpep_dropoff_datetime"] - trips["tpep_pickup_datetime"]
).dt.total_seconds() / 60

initial_checks = pd.Series({
    "Missing pickup timestamps": trips["tpep_pickup_datetime"].isna().sum(),
    "Missing drop-off timestamps": trips["tpep_dropoff_datetime"].isna().sum(),
    "Zero or negative trip durations": duration_minutes.le(0).sum(),
    "Zero or negative trip distances": trips["trip_distance"].le(0).sum(),
    "Negative fare amounts": trips["fare_amount"].lt(0).sum()
})

display(initial_checks)

Missing pickup timestamps               0
Missing drop-off timestamps             0
Zero or negative trip durations      2051
Zero or negative trip distances     90893
Negative fare amounts              144118
dtype: int64

### 5. Check duplicates, pickup dates, and location references

These checks identify exact duplicate rows, nonmissing pickup timestamps
outside January 2025, and pickup or drop-off location IDs that do not
match the taxi-zone lookup table.

Duplicate counts represent additional occurrences after the first row.
Flagged records are retained for investigation.

In [5]:
# Identify additional occurrences of exact duplicate rows.
duplicate_rows = trips.duplicated(keep="first")

# Use an inclusive start and exclusive end for January 2025.
pickup = trips["tpep_pickup_datetime"]
january_start = pd.Timestamp("2025-01-01")
february_start = pd.Timestamp("2025-02-01")

outside_january = pickup.notna() & (
    (pickup < january_start) | (pickup >= february_start)
)

# Check references against the actual lookup IDs.
valid_location_ids = zones["LocationID"].dropna()

unmatched_pickup = ~trips["PULocationID"].isin(valid_location_ids)
unmatched_dropoff = ~trips["DOLocationID"].isin(valid_location_ids)

checks = {
    "Exact duplicate rows beyond first occurrence": duplicate_rows,
    "Pickup timestamps outside January 2025": outside_january,
    "Pickup IDs missing or absent from lookup": unmatched_pickup,
    "Drop-off IDs missing or absent from lookup": unmatched_dropoff,
}

quality_report = pd.DataFrame([
    {
        "check": name,
        "flagged_count": int(mask.sum()),
        "flagged_percent": round(mask.mean() * 100, 4),
    }
    for name, mask in checks.items()
])

display(quality_report)

# Verify that the lookup can support a many-to-one join.
print(
    "Duplicate nonmissing LocationIDs in lookup:",
    valid_location_ids.duplicated().sum()
)
print("Missing LocationIDs in lookup:", zones["LocationID"].isna().sum())

# Show a few examples for investigation.
for name, mask in checks.items():
    print(f"\n{name}")
    display(trips.loc[mask].head(5))

,check,flagged_count,flagged_percent
0,Exact duplicate rows beyond first occurrence,0,0.0000
1,Pickup timestamps outside January 2025,22,0.0006
2,Pickup IDs missing or absent from lookup,0,0.0000
3,Drop-off IDs missing or absent from lookup,0,0.0000


Duplicate nonmissing LocationIDs in lookup: 0
Missing LocationIDs in lookup: 0

Exact duplicate rows beyond first occurrence


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee



Pickup timestamps outside January 2025


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
605,2,2024-12-31 23:30:03,2024-12-31 23:43:02,1.0,3.00,1.0,N,246,13,1,16.3,1.0,0.5,5.32,0.0,1.0,26.62,2.5,0.0,0.0
687,2,2024-12-31 23:31:38,2024-12-31 23:41:48,1.0,1.03,1.0,N,43,140,2,10.7,1.0,0.5,0.00,0.0,1.0,15.70,2.5,0.0,0.0
688,2,2024-12-31 23:46:38,2025-01-01 00:03:03,1.0,3.95,1.0,N,229,24,2,19.1,1.0,0.5,0.00,0.0,1.0,24.10,2.5,0.0,0.0
861,2,2024-12-31 23:56:19,2025-01-01 00:11:19,6.0,2.28,1.0,N,68,107,1,14.9,1.0,0.5,3.98,0.0,1.0,23.88,2.5,0.0,0.0
1108,2,2024-12-31 23:55:37,2025-01-01 00:01:26,1.0,1.12,1.0,N,56,56,2,7.9,1.0,0.5,0.00,0.0,1.0,10.40,0.0,0.0,0.0



Pickup IDs missing or absent from lookup


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee



Drop-off IDs missing or absent from lookup


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee


### Validation Findings

No exact duplicate trip rows were found. Twenty-two records had pickup timestamps outside January 2025, representing approximately 0.0006% of the dataset. The displayed examples include December 31, 2024 pickups, some of which ended after midnight on January 1. 

All pickup and drop-off location IDs matched entries in the taxi-zone lookup table. The lookup contained no missing or duplicate LocationIDs, supporting many-to-one joins with the trip records. A successful match does not necessarily indicate a specific known geographic location.

The flagged records remain unchanged in the raw dataset. For January analytics, the reporting period will be defined using pickup timestamps. In the next processing stage, records with pickup timestamps outside January 2025 will be excluded from January-specific summaries and retained separately for review.  

### 6. Consolidated Data Quality Report

This report combines the validation checks and documents the planned
action for each issue. Raw records remain unchanged.

Missing timestamps prevent time-based analysis. Nonpositive durations
are excluded from duration and speed calculations. Nonpositive distances
are excluded from distance and speed calculations. Negative fares are
retained for review before use in fare summaries.

Pickups outside January are excluded from January-specific summaries.
Unmatched location IDs prevent geographic summaries but do not
automatically exclude records from other analyses. Exact duplicate
rows require review before removal.

Missing values in supplementary fields are reported without automatically
excluding the entire record. Counts may overlap because a record can
trigger multiple rules. All percentages use the total raw row count.

In [6]:
rules = [
    (
        "Missing pickup timestamp",
        trips["tpep_pickup_datetime"].isna(),
        "Exclude from time-based summaries"
    ),
    (
        "Missing drop-off timestamp",
        trips["tpep_dropoff_datetime"].isna(),
        "Exclude from duration and speed calculations"
    ),
    (
        "Nonpositive trip duration",
        duration_minutes.le(0),
        "Exclude from duration and speed calculations"
    ),
    (
        "Nonpositive trip distance",
        trips["trip_distance"].le(0),
        "Exclude from distance and speed calculations"
    ),
    (
        "Negative fare",
        trips["fare_amount"].lt(0),
        "Review before use in fare summaries"
    ),
    (
        "Exact duplicate beyond first occurrence",
        duplicate_rows,
        "Review before removal"
    ),
    (
        "Pickup outside January 2025",
        outside_january,
        "Exclude from January-specific summaries"
    ),
    (
        "Pickup ID missing or absent from lookup",
        unmatched_pickup,
        "Exclude from pickup geographic summaries"
    ),
    (
        "Drop-off ID missing or absent from lookup",
        unmatched_dropoff,
        "Exclude from drop-off geographic summaries"
    ),
]

# Report missing values in each supplementary field separately.
supplementary_fields = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee",
]

for column in supplementary_fields:
    rules.append((
        f"Missing {column}",
        trips[column].isna(),
        "Retain record; flag field as unavailable"
    ))

quality_report = pd.DataFrame([
    {
        "rule": name,
        "flagged_count": int(mask.sum()),
        "flagged_percent": round(mask.mean() * 100, 4),
        "planned_action": action,
    }
    for name, mask, action in rules
])

display(quality_report)

# Count distinct records triggering at least one listed rule.
any_flag = pd.Series(False, index=trips.index)

for _, mask, _ in rules:
    any_flag |= mask

print(f"Total raw records: {len(trips):,}")
print(f"Records with at least one listed flag: {any_flag.sum():,}")
print(f"Percent with at least one listed flag: {any_flag.mean():.2%}")

# Save the report separately from the raw data.
report_dir = Path("../reports")
report_dir.mkdir(parents=True, exist_ok=True)

report_path = report_dir / "quality_report_2025-01.csv"
quality_report.to_csv(report_path, index=False)

print(f"Report saved to: {report_path.resolve()}")

,rule,flagged_count,flagged_percent,planned_action
0,Missing pickup timestamp,0,0.0000,Exclude from time-based summaries
1,Missing drop-off timestamp,0,0.0000,Exclude from duration and speed calculations
2,Nonpositive trip duration,2051,0.0590,Exclude from duration and speed calculations
3,Nonpositive trip distance,90893,2.6155,Exclude from distance and speed calculations
4,Negative fare,144118,4.1470,Review before use in fare summaries
5,Exact duplicate beyond first occurrence,0,0.0000,Review before removal
6,Pickup outside January 2025,22,0.0006,Exclude from January-specific summaries
7,Pickup ID missing or absent from lookup,0,0.0000,Exclude from pickup geographic summaries
8,Drop-off ID missing or absent from lookup,0,0.0000,Exclude from drop-off geographic summaries
9,Missing passenger_count,540149,15.5428,Retain record; flag field as unavailable


Total raw records: 3,475,226
Records with at least one listed flag: 635,232
Percent with at least one listed flag: 18.28%
Report saved to: C:\Users\ferri\anaconda_projects\DSC680\Week_09\NYC_Taxi\reports\quality_report_2025-01.csv


### 7. Investigate Missing-Value Patterns and Extreme Values

This section checks whether the five supplementary fields are missing
in the same records and examines the largest distances, fares, and
trip durations.

These checks support future processing decisions. No records are
removed, and no upper-limit thresholds are applied at this stage.

In [7]:
# Check whether supplementary fields are missing together.
missing_flags = trips[supplementary_fields].isna()

all_five_missing = missing_flags.all(axis=1)
any_field_missing = missing_flags.any(axis=1)
partially_missing = any_field_missing & ~all_five_missing

print(f"All five fields missing: {all_five_missing.sum():,}")
print(f"Some, but not all, fields missing: {partially_missing.sum():,}")

# Explore whether missingness differs by vendor.
missing_by_vendor = (
    trips.assign(all_five_missing=all_five_missing)
    .groupby("VendorID")
    .agg(
        total_records=("all_five_missing", "size"),
        records_all_five_missing=("all_five_missing", "sum")
    )
)

missing_by_vendor["percent_all_five_missing"] = (
    missing_by_vendor["records_all_five_missing"]
    / missing_by_vendor["total_records"] * 100
).round(2)

display(missing_by_vendor)

# Examine upper percentiles before choosing any thresholds.
numeric_review = pd.DataFrame({
    "trip_distance": trips["trip_distance"],
    "fare_amount": trips["fare_amount"],
    "duration_minutes": duration_minutes
})

display(
    numeric_review.quantile([0.50, 0.95, 0.99, 0.999, 1.0])
    .rename_axis("quantile")
)

# Display records with the largest values.
review_columns = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "PULocationID",
    "DOLocationID"
]

for column in ["trip_distance", "fare_amount"]:
    print(f"\nFive largest values: {column}")
    display(trips.nlargest(5, column)[review_columns])

print("\nFive longest trip durations")
longest_indices = duration_minutes.nlargest(5).index

display(
    trips.loc[longest_indices, review_columns]
    .assign(duration_minutes=duration_minutes.loc[longest_indices])
)

All five fields missing: 540,149
Some, but not all, fields missing: 0


,total_records,records_all_five_missing,percent_all_five_missing
VendorID,,,
1,753671,88204,11.7
2,2719860,451456,16.6
6,489,489,100.0
7,1206,0,0.0


,trip_distance,fare_amount,duration_minutes
quantile,,,
0.500,1.67000,12.11,11.700000
0.950,11.83000,52.00,36.200000
0.990,19.50000,72.30,58.916667
0.999,28.85775,132.00,98.262917
1.000,276423.57000,863372.12,5626.316667



Five largest values: trip_distance


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,total_amount,PULocationID,DOLocationID
3275762,2,2025-01-22 21:04:00,2025-01-22 21:13:00,276423.57,-4.75,5.00,161,170
3188501,2,2025-01-19 14:51:00,2025-01-19 14:58:00,276099.95,9.13,13.88,224,233
3240337,2,2025-01-21 09:19:00,2025-01-21 09:45:00,222167.49,31.19,35.94,52,144
3112173,2,2025-01-16 10:47:00,2025-01-16 11:11:00,206137.99,24.89,29.64,263,48
3374854,2,2025-01-26 17:46:00,2025-01-26 17:59:00,202771.63,10.10,14.85,50,186



Five largest values: fare_amount


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,total_amount,PULocationID,DOLocationID
1780915,1,2025-01-20 12:07:18,2025-01-20 12:12:42,1.60,863372.12,863380.37,138,8
1404958,2,2025-01-16 12:23:14,2025-01-17 14:41:56,255.33,2450.90,2506.71,161,265
1832645,2,2025-01-21 05:17:57,2025-01-21 08:19:45,188.88,1309.20,1311.70,247,265
2358832,2,2025-01-26 00:04:04,2025-01-26 00:04:42,0.00,950.00,953.50,142,142
2358836,2,2025-01-26 00:22:47,2025-01-26 00:22:55,0.00,950.00,951.00,226,226



Five longest trip durations


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,total_amount,PULocationID,DOLocationID,duration_minutes
1858816,2,2025-01-21 12:23:41,2025-01-25 10:10:00,6.55,-28.2,-36.45,70,70,5626.316667
1858817,2,2025-01-21 12:23:41,2025-01-25 10:10:00,6.55,28.2,36.45,70,129,5626.316667
414374,2,2025-01-06 02:12:11,2025-01-09 13:54:19,91.51,336.9,339.40,218,193,5022.133333
2460536,2,2025-01-27 11:58:55,2025-01-30 09:42:55,0.00,3.0,4.50,207,226,4184.000000
898201,2,2025-01-11 08:04:00,2025-01-13 16:49:28,0.04,70.0,76.50,132,226,3405.466667


### Investigation Findings

The five supplementary fields are missing together in 540,149 records.
No partially missing combinations were found. Missingness varies across
vendors, although this analysis does not establish its cause.

The maximum distances, fares, and durations differ substantially from
their 99.9th percentiles. For example, the largest recorded distance is
276,423.57 miles for a nine-minute trip, and the largest fare is
$863,372.12 for a 1.6-mile trip. These records require review before
use in distance, speed, or fare summaries.

The longest recorded duration is approximately 3.9 days. Duration
outliers also require review before analytical use.

Raw records remain unchanged. Upper-limit rules will be documented
as project assumptions and evaluated before filtering.